In [2]:
%run ./tmdbapi.ipynb

In [3]:
import requests
url='https://api.themoviedb.org/3/genre/movie/list?api_key='+API_key
response=requests.get(url)
response.json()


{'genres': [{'id': 28, 'name': 'Action'},
  {'id': 12, 'name': 'Adventure'},
  {'id': 16, 'name': 'Animation'},
  {'id': 35, 'name': 'Comedy'},
  {'id': 80, 'name': 'Crime'},
  {'id': 99, 'name': 'Documentary'},
  {'id': 18, 'name': 'Drama'},
  {'id': 10751, 'name': 'Family'},
  {'id': 14, 'name': 'Fantasy'},
  {'id': 36, 'name': 'History'},
  {'id': 27, 'name': 'Horror'},
  {'id': 10402, 'name': 'Music'},
  {'id': 9648, 'name': 'Mystery'},
  {'id': 10749, 'name': 'Romance'},
  {'id': 878, 'name': 'Science Fiction'},
  {'id': 10770, 'name': 'TV Movie'},
  {'id': 53, 'name': 'Thriller'},
  {'id': 10752, 'name': 'War'},
  {'id': 37, 'name': 'Western'}]}

In [54]:
def get_data(pageno,year,genre):
  url='https://api.themoviedb.org/3/discover/movie?api_key='+API_key+'&primary_release_year.gte='+str(year)+'&sort_by=popularity.desc&with_genres='+str(genre)+'&page='+str(pageno)
  response=requests.get(url)
  if response.status_code==200:
    data=response.json()
    return data['results']
  else:
    return 'error'

In [55]:
import pandas as pd
df=pd.DataFrame(columns=['Movie ID','Movie Name'])

In [56]:
import time
pageno=1
while(len(df)<300):

  query_results=get_data(pageno,2000,35)
  if query_results !='error':
    for movie in query_results:
      df.loc[len(df)]=[movie['id'],movie['title']]

  pageno+=1
  time.sleep(0.25)



In [57]:
len(df['Movie ID'].unique())

253

In [8]:
def get_similar_movie(movieid):
  url='https://api.themoviedb.org/3/movie/'+str(movieid)+'/similar?api_key='+API_key
  response=requests.get(url)
  if response.status_code==200:
    return response.json()['results']
  else:
    return 'error'




In [9]:
get_similar_movie(1204680)

[{'adult': False,
  'backdrop_path': '/nv6F6tz7r61DUhE7zgHwLJFcTYp.jpg',
  'genre_ids': [35, 10749, 80],
  'id': 974635,
  'title': 'Hit Man',
  'original_language': 'en',
  'original_title': 'Hit Man',
  'overview': 'A mild-mannered professor moonlighting as a fake hit man in police stings ignites a chain reaction of trouble when he falls for a potential client.',
  'popularity': 14.3166,
  'poster_path': '/oil3EZwKFp3CWxZnfGfGglesvm9.jpg',
  'release_date': '2024-05-16',
  'softcore': False,
  'video': False,
  'vote_average': 6.78,
  'vote_count': 1740},
 {'adult': False,
  'backdrop_path': None,
  'genre_ids': [35, 10749],
  'id': 192280,
  'title': "Helen's Marriage",
  'original_language': 'en',
  'original_title': "Helen's Marriage",
  'overview': 'Helen and Tom are desperate to get married. After a foiled elopement attempt, the couple hatches a “fake wedding” scheme to trick Helen’s protective parents into allowing their marriage.',
  'popularity': 0.8437,
  'poster_path': '/3O

In [10]:
df1=pd.DataFrame(columns=['Movie ID','Similar Movie ID'])
for movieid in df['Movie ID']:
  query_results=get_similar_movie(movieid)
  if query_results!='error':
    for i in range(min(5,len(query_results))):
      df1.loc[len(df1)]=[movieid,query_results[i]['id']]

In [11]:
df1.head(10)

,Movie ID,Similar Movie ID
0,1204680,974635
1,1204680,192280
2,1204680,192305
3,1204680,278927
4,1204680,18943
5,1393326,601083
6,1393326,601099
7,1393326,974626
8,1393326,1368494
9,1393326,595143


In [58]:
def get_imdb_id(movieid):
  url='https://api.themoviedb.org/3/movie/'+str(movieid)+'/external_ids?api_key='+str(API_key)
  response=requests.get(url)
  if response.status_code==200:
    return response.json()
  else:
    return 'error'

In [59]:
get_imdb_id(1204680)

{'id': 1204680,
 'imdb_id': 'tt1756855',
 'wikidata_id': 'Q106392679',
 'facebook_id': 'coyoteacmemovie',
 'instagram_id': 'coyoteacmemovie',
 'twitter_id': 'CoyoteACMEMovie'}

In [60]:
df['IMDB_ID']=None

In [61]:
for i,movieid in enumerate(df['Movie ID']):
  query_results=get_imdb_id(movieid)
  if query_results!='error':
    df.loc[i,'IMDB_ID']=query_results['imdb_id']

In [62]:
IMDB_data=pd.read_csv('/content/title.ratings.tsv',sep='\t')

In [63]:
df=df.merge(IMDB_data,how='left',left_on='IMDB_ID',right_on='tconst')

In [64]:
df['IMDB_URL']='https://www.imdb.com/title/'+df['IMDB_ID'].fillna(' ')+'/'

In [65]:
df

,Movie ID,Movie Name,IMDB_ID,tconst,averageRating,numVotes,IMDB_URL
0,1204680,Coyote vs. Acme,tt1756855,tt1756855,7.6,13354.0,https://www.imdb.com/title/tt1756855/
1,1393326,Ghost in the Cell,tt9000310,tt9000310,6.7,2604.0,https://www.imdb.com/title/tt9000310/
2,1137844,Mayday,tt28014327,tt28014327,6.9,3388.0,https://www.imdb.com/title/tt28014327/
3,1084244,Toy Story 5,tt29355505,tt29355505,7.4,77768.0,https://www.imdb.com/title/tt29355505/
4,1315772,Minions & Monsters,tt32890033,tt32890033,6.3,30236.0,https://www.imdb.com/title/tt32890033/
...,...,...,...,...,...,...,...
295,49524,R.I.P.D.,tt0790736,tt0790736,5.6,152602.0,https://www.imdb.com/title/tt0790736/
296,19908,Zombieland,tt1156398,tt1156398,7.5,673782.0,https://www.imdb.com/title/tt1156398/
297,585511,Luck,tt7214954,tt7214954,6.5,36302.0,https://www.imdb.com/title/tt7214954/
298,840430,The Holdovers,tt14849194,tt14849194,7.9,273438.0,https://www.imdb.com/title/tt14849194/


In [66]:
df.rename(columns={'averageRating':'IMDB_Ratings','numVotes':'IMDB Votes'},inplace=True)
df.drop('tconst',axis=1)

,Movie ID,Movie Name,IMDB_ID,IMDB_Ratings,IMDB Votes,IMDB_URL
0,1204680,Coyote vs. Acme,tt1756855,7.6,13354.0,https://www.imdb.com/title/tt1756855/
1,1393326,Ghost in the Cell,tt9000310,6.7,2604.0,https://www.imdb.com/title/tt9000310/
2,1137844,Mayday,tt28014327,6.9,3388.0,https://www.imdb.com/title/tt28014327/
3,1084244,Toy Story 5,tt29355505,7.4,77768.0,https://www.imdb.com/title/tt29355505/
4,1315772,Minions & Monsters,tt32890033,6.3,30236.0,https://www.imdb.com/title/tt32890033/
...,...,...,...,...,...,...
295,49524,R.I.P.D.,tt0790736,5.6,152602.0,https://www.imdb.com/title/tt0790736/
296,19908,Zombieland,tt1156398,7.5,673782.0,https://www.imdb.com/title/tt1156398/
297,585511,Luck,tt7214954,6.5,36302.0,https://www.imdb.com/title/tt7214954/
298,840430,The Holdovers,tt14849194,7.9,273438.0,https://www.imdb.com/title/tt14849194/


In [21]:
df

,Movie ID,Movie Name,IMDB_ID,tconst,IMDB_Ratings,IMDB Votes,IMDB_URL
0,1204680,Coyote vs. Acme,tt1756855,tt1756855,7.6,13354,https://www.imdb.com/title/tt1756855/
1,1393326,Ghost in the Cell,tt9000310,tt9000310,6.7,2604,https://www.imdb.com/title/tt9000310/
2,1137844,Mayday,tt28014327,tt28014327,6.9,3388,https://www.imdb.com/title/tt28014327/
3,1084244,Toy Story 5,tt29355505,tt29355505,7.4,77768,https://www.imdb.com/title/tt29355505/
4,1315772,Minions & Monsters,tt32890033,tt32890033,6.3,30236,https://www.imdb.com/title/tt32890033/
...,...,...,...,...,...,...,...
295,533535,Deadpool & Wolverine,tt6263850,tt6263850,7.5,592107,https://www.imdb.com/title/tt6263850/
296,862,Toy Story,tt0114709,tt0114709,8.3,1197298,https://www.imdb.com/title/tt0114709/
297,1612018,Jackass: Best and Last,tt39316472,tt39316472,6.2,11797,https://www.imdb.com/title/tt39316472/
298,1327819,Hoppers,tt26443616,tt26443616,7.2,83956,https://www.imdb.com/title/tt26443616/
